# Transformasi Matriks

**Blok Utama** (biru): posisi asli dari titik GeoGebra A(2,3) B(2,4) C(3,4) D(3,3)  
**Blok Cermin** (merah): refleksi terhadap sumbu X  

Transformasi: $T = \begin{bmatrix} 1 & 0 \\ 0 & s \end{bmatrix}$ dengan $s$ turun dari $1 \to 0$



In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.patches as patches
from matplotlib.lines import Line2D
from IPython.display import HTML, display

In [2]:
# ── Definisi blok dari titik GeoGebra ───────────────────────────
# Blok Utama  : A(2,3) B(2,4) C(3,4) D(3,3)  → x:[2,3]  y:[3,4]
# Blok Cermin : refleksi sumbu X              → x:[2,3]  y:[-4,-3]

X0, X1 = 2, 3      # batas x (sama untuk kedua blok)
Y0, Y1 = 3, 4      # batas y blok utama (posisi awal)
W = X1 - X0        # lebar blok
H = Y1 - Y0        # tinggi blok

FRAMES  = 150
PAUSE_F = 25

# ── Setup figure ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 6))
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

ax.set_xlim(-3, 7)
ax.set_ylim(-5, 5)
ax.set_aspect('equal')
ax.axhline(0, color='black', linewidth=1.0, zorder=2)
ax.axvline(0, color='black', linewidth=1.0, zorder=2)
ax.grid(True, color='#cccccc', linewidth=0.5, linestyle='-')
ax.tick_params(labelsize=9, colors='#333')
for spine in ax.spines.values():
    spine.set_edgecolor('#aaaaaa')

ax.set_title('Animasi Pergerakan Blok (Klik grafik untuk Pause/Play)',
             fontsize=11, color='#222', pad=10)

# ── Blok Utama (biru) ───────────────────────────────────────────
rect_main = patches.Rectangle(
    (X0, Y0), W, H,
    linewidth=1.8, edgecolor='#1a6fb5', facecolor='none', zorder=5
)
ax.add_patch(rect_main)
dots_main, = ax.plot([], [], 'o', color='#1a6fb5', markersize=6, zorder=6)
lbl_main = ['A','B','C','D']
texts_main = [
    ax.text(0, 0, lbl, color='#1a6fb5', fontsize=8, fontweight='bold',
            ha='left', va='bottom', zorder=7)
    for lbl in lbl_main
]

# ── Blok Cermin (merah) ─────────────────────────────────────────
rect_mirror = patches.Rectangle(
    (X0, -Y1), W, H,
    linewidth=1.8, edgecolor='#c0392b', facecolor='none', zorder=5
)
ax.add_patch(rect_mirror)
dots_mirror, = ax.plot([], [], 'o', color='#c0392b', markersize=6, zorder=6)
lbl_mirror = ["A'","B'","C'","D'"]
texts_mirror = [
    ax.text(0, 0, lbl, color='#c0392b', fontsize=8, fontweight='bold',
            ha='left', va='bottom', zorder=7)
    for lbl in lbl_mirror
]

# ── Legend ──────────────────────────────────────────────────────
legend_elements = [
    Line2D([0],[0], color='#1a6fb5', linewidth=2, label='Blok Utama'),
    Line2D([0],[0], color='#c0392b', linewidth=2, label='Blok Cermin'),
]
ax.legend(handles=legend_elements, loc='upper right', fontsize=9,
          framealpha=0.9, edgecolor='#cccccc')

# ── Info teks matriks ───────────────────────────────────────────
mat_txt = ax.text(-2.8, 4.5, '', fontsize=9, color='#222',
                  fontfamily='monospace',
                  bbox=dict(boxstyle='round,pad=0.4', facecolor='#f0f4ff',
                            edgecolor='#aaaaaa', alpha=0.9))

# ── State pause/play ────────────────────────────────────────────
state = {'paused': False}

def on_click(event):
    if event.inaxes == ax:
        state['paused'] = not state['paused']

fig.canvas.mpl_connect('button_press_event', on_click)

# ── Easing: 1.0 → 0.0 → 1.0 (bolak-balik) ─────────────────────
def get_scale(frame):
    if frame < PAUSE_F:
        return 1.0
    if frame >= FRAMES - PAUSE_F:
        return 1.0
    mid = FRAMES // 2
    if frame <= mid:
        t = (frame - PAUSE_F) / (mid - PAUSE_F)
        t = 3*t**2 - 2*t**3
        return 1.0 - t
    else:
        t = (frame - mid) / (FRAMES - PAUSE_F - mid)
        t = 3*t**2 - 2*t**3
        return t

current_frame = [0]

def animate(frame):
    if state['paused']:
        frame = current_frame[0]
    else:
        current_frame[0] = frame

    s = get_scale(frame)

    # Blok utama: y dikompres mendekati sumbu X
    y0m = Y0 * s
    y1m = Y1 * s
    rect_main.set_y(y0m)
    rect_main.set_height(max(y1m - y0m, 1e-6))

    # 4 sudut: BL, BR, TR, TL
    cx = [X0, X1, X1, X0]
    cy_main = [y0m, y0m, y1m, y1m]
    dots_main.set_data(cx, cy_main)
    offsets = [(-0.15,-0.15),(0.05,-0.15),(0.05,0.05),(-0.15,0.05)]
    for tx, px, py, (dx, dy) in zip(texts_main, cx, cy_main, offsets):
        tx.set_position((px + dx, py + dy))

    # Blok cermin: refleksi
    y0r = -Y1 * s
    y1r = -Y0 * s
    rect_mirror.set_y(y0r)
    rect_mirror.set_height(max(y1r - y0r, 1e-6))

    cy_mirror = [y0r, y0r, y1r, y1r]
    dots_mirror.set_data(cx, cy_mirror)
    offsets_m = [(-0.15,-0.20),(0.05,-0.20),(0.05,0.05),(-0.15,0.05)]
    for tx, px, py, (dx, dy) in zip(texts_mirror, cx, cy_mirror, offsets_m):
        tx.set_position((px + dx, py + dy))

    mat_txt.set_text(f'T = [1   0 ]\n     [0  {s:.2f}]\n\nScale Y: {s:.3f}')

    return (rect_main, rect_mirror, dots_main, dots_mirror,
            mat_txt, *texts_main, *texts_mirror)

ani = animation.FuncAnimation(
    fig, animate, frames=FRAMES,
    interval=40, blit=True, repeat=True
)

plt.tight_layout()
display(HTML(ani.to_jshtml()))
plt.close()